> **Niveau 🟡 moyen — les étapes sont données en commentaire, écrivez le code**

# Notebook 2 — Du patient à AlphaFold

On dispose de la séquence codante de **quatre gènes exprimés dans le globule rouge**, chez un
sujet de référence et chez le patient :

| Gène | Protéine |
|---|---|
| `HBA1` | alpha-globine |
| `HBB` | bêta-globine |
| `HBD` | delta-globine |
| `HBG1` | gamma-globine (hémoglobine fœtale) |

Objectifs :

1. trouver **lequel de ces gènes est muté**, quelle base change, et quel acide aminé ;
2. dire si ce changement d'acide aminé est important ;
3. préparer les séquences de l'hémoglobine normale et de celle du patient, les soumettre au
   **serveur AlphaFold**, et comparer les deux structures prédites.

**Mode d'emploi**
- `Maj + Entrée` exécute une cellule et passe à la suivante.
- Exécutez les cellules **dans l'ordre**, de haut en bas.
- Les cellules **✔️ Vérification** ne se modifient pas : elles affichent ✅ quand votre code est juste.

## 0. Charger les séquences

Exécutez la cellule ci-dessous : elle crée deux fichiers FASTA — la référence et le patient — et
range leur contenu dans deux dictionnaires, `REFERENCE` et `PATIENT`. Elle redéfinit aussi les
trois fonctions du notebook 1 : `transcrire()`, `codons()` et `traduire()`.

In [ ]:
#@title ▶️ Exécutez cette cellule pour charger les séquences (ne pas modifier)
# Séquences codantes de référence (RefSeq) de quatre gènes exprimés dans le globule rouge,
# et les mêmes gènes séquencés chez le patient.

FASTA_REFERENCE = """>HBA1 alpha-globine — NM_000558.5
ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCG
CACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACC
AAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGC
AAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCG
CTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAG
CTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCT
GCGGTGCACGCCTCCCTGGACAAGTTCCTGGCTTCTGTGAGCACCGTGCTGACCTCCAAA
TACCGTTAA
>HBB bêta-globine — NM_000518.5
ATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC
GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC
AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCCCACAAGTATCACTAA
>HBD delta-globine — NM_000519.4
ATGGTGCATCTGACTCCTGAGGAGAAGACTGCTGTCAATGCCCTGTGGGGCAAAGTGAAC
GTGGATGCAGTTGGTGGTGAGGCCCTGGGCAGATTACTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCTCTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAGGTGCTAGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACTTTTTCTCAGCTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCTTGGGCAATGTGCTGGTGTGTGTGCTGGCCCGCAACTTTGGC
AAGGAATTCACCCCACAAATGCAGGCTGCCTATCAGAAGGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCTCACAAGTACCATTGA
>HBG1 gamma-globine (hémoglobine fœtale) — NM_000559.3
ATGGGTCATTTCACAGAGGAGGACAAGGCTACTATCACAAGCCTGTGGGGCAAGGTGAAT
GTGGAAGATGCTGGAGGAGAAACCCTGGGAAGGCTCCTGGTTGTCTACCCATGGACCCAG
AGGTTCTTTGACAGCTTTGGCAACCTGTCCTCTGCCTCTGCCATCATGGGCAACCCCAAA
GTCAAGGCACATGGCAAGAAGGTGCTGACTTCCTTGGGAGATGCCACAAAGCACCTGGAT
GATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGAT
CCTGAGAACTTCAAGCTCCTGGGAAATGTGCTGGTGACCGTTTTGGCAATCCATTTCGGC
AAAGAATTCACCCCTGAGGTGCAGGCTTCCTGGCAGAAGATGGTGACTGCAGTGGCCAGT
GCCCTGTCCTCCAGATACCACTGA
"""

FASTA_PATIENT = """>HBA1 alpha-globine — NM_000558.5
ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCG
CACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACC
AAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGC
AAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCG
CTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAG
CTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCT
GCGGTGCACGCCTCCCTGGACAAGTTCCTGGCTTCTGTGAGCACCGTGCTGACCTCCAAA
TACCGTTAA
>HBB bêta-globine — NM_000518.5
ATGGTGCATCTGACTCCTGTGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC
GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC
AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCCCACAAGTATCACTAA
>HBD delta-globine — NM_000519.4
ATGGTGCATCTGACTCCTGAGGAGAAGACTGCTGTCAATGCCCTGTGGGGCAAAGTGAAC
GTGGATGCAGTTGGTGGTGAGGCCCTGGGCAGATTACTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCTCTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAGGTGCTAGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACTTTTTCTCAGCTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCTTGGGCAATGTGCTGGTGTGTGTGCTGGCCCGCAACTTTGGC
AAGGAATTCACCCCACAAATGCAGGCTGCCTATCAGAAGGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCTCACAAGTACCATTGA
>HBG1 gamma-globine (hémoglobine fœtale) — NM_000559.3
ATGGGTCATTTCACAGAGGAGGACAAGGCTACTATCACAAGCCTGTGGGGCAAGGTGAAT
GTGGAAGATGCTGGAGGAGAAACCCTGGGAAGGCTCCTGGTTGTCTACCCATGGACCCAG
AGGTTCTTTGACAGCTTTGGCAACCTGTCCTCTGCCTCTGCCATCATGGGCAACCCCAAA
GTCAAGGCACATGGCAAGAAGGTGCTGACTTCCTTGGGAGATGCCACAAAGCACCTGGAT
GATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGAT
CCTGAGAACTTCAAGCTCCTGGGAAATGTGCTGGTGACCGTTTTGGCAATCCATTTCGGC
AAAGAATTCACCCCTGAGGTGCAGGCTTCCTGGCAGAAGATGGTGACTGCAGTGGCCAGT
GCCCTGTCCTCCAGATACCACTGA
"""

with open("reference.fasta", "w") as f:
    f.write(FASTA_REFERENCE)
with open("patient.fasta", "w") as f:
    f.write(FASTA_PATIENT)


def lire_multifasta(nom_fichier):
    """Lit un fichier FASTA contenant plusieurs séquences.
    Renvoie un dictionnaire {nom du gène: séquence}."""
    sequences = {}
    nom = None
    with open(nom_fichier) as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne.startswith(">"):
                nom = ligne[1:].split()[0]
                sequences[nom] = ""
            elif ligne:
                sequences[nom] = sequences[nom] + ligne
    return sequences


CODE_GENETIQUE = {
    "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L",
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S",
    "UAU": "Y", "UAC": "Y", "UAA": "*", "UAG": "*",
    "UGU": "C", "UGC": "C", "UGA": "*", "UGG": "W",
    "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AUU": "I", "AUC": "I", "AUA": "I", "AUG": "M",
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGU": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}


# Les trois fonctions écrites au notebook 1
def transcrire(adn):
    """Renvoie l'ARN messager : la séquence de adn avec chaque T remplacé par U."""
    return adn.replace("T", "U")


def codons(arn):
    """Découpe arn en triplets consécutifs à partir de l'index 0."""
    liste = []
    for i in range(0, len(arn), 3):
        liste.append(arn[i:i + 3])
    return liste


def traduire(arn):
    """Traduit arn en protéine (code à une lettre), jusqu'au premier codon stop exclu."""
    proteine = ""
    for codon in codons(arn):
        acide_amine = CODE_GENETIQUE[codon]
        if acide_amine == "*":
            break
        proteine = proteine + acide_amine
    return proteine


REFERENCE = lire_multifasta("reference.fasta")
PATIENT = lire_multifasta("patient.fasta")
print(len(REFERENCE), "gènes chargés ✅")

Un même fichier FASTA peut contenir plusieurs séquences à la suite — ici, les quatre gènes.
La fonction `lire_multifasta()` le transforme en **dictionnaire** : à chaque nom de gène
correspond sa séquence, en une seule chaîne de caractères.

In [ ]:
# le fichier, tel qu'il est écrit sur le disque (ses 5 premières lignes)
for ligne in FASTA_REFERENCE.splitlines()[:5]:
    print(ligne)
print("...")

print()

# ce que Python en a fait : un dictionnaire
print("les clés  :", list(REFERENCE))
print("REFERENCE['HBB'] :", REFERENCE["HBB"][:40], "...")
print("type      :", type(REFERENCE["HBB"]))

## 1. Quel gène est muté ?

Les quatre gènes du patient ont été séquencés. Trois sont identiques à la référence, un seul
diffère — c'est celui-là qu'il faut trouver.

Pour parcourir un dictionnaire : `for cle in mon_dictionnaire:` donne ses clés, une à une. Deux
chaînes de caractères se comparent directement : `"ACGT" == "ACGT"` vaut `True`.

In [ ]:
# 1. créer une variable gene_mute, initialisée à None
# 2. boucle for sur les noms de gènes de REFERENCE
# 3. comparer REFERENCE[nom] et PATIENT[nom] avec == : afficher "identique" ou "DIFFÉRENT"
# 4. quand les deux diffèrent, ranger le nom dans gene_mute
# 5. afficher gene_mute
# 6. définir adn_wt = REFERENCE[gene_mute] et adn_patient = PATIENT[gene_mute]
pass  # ← remplacez cette ligne par votre code

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if gene_mute == "HBB":
    print("✅ le gène muté est HBB — la bêta-globine, une des deux chaînes de l'hémoglobine")
else:
    print("❌ gène trouvé :", gene_mute, "— reprenez la comparaison, gène par gène")

if adn_wt == REFERENCE["HBB"] and adn_patient == PATIENT["HBB"]:
    print("✅ adn_wt et adn_patient contiennent bien les séquences de ce gène")
else:
    print("❌ adn_wt et adn_patient doivent valoir REFERENCE[gene_mute] et PATIENT[gene_mute]")

## 2. Quelle base ?

À l'œil, 444 lettres, c'est trop. On écrit une fonction `trouver_mutations(ref, patient)` qui
parcourt les deux séquences position par position et renvoie la **liste des différences**,
chacune sous la forme `(index, base_ref, base_patient)`.

Exemple : `trouver_mutations("ACGT", "ACCT")` → `[(2, "G", "C")]`

⚠️ Python numérote à partir de **0** ; les biologistes numérotent les nucléotides à partir de **1**.
Le nucléotide n°1 est à l'index 0.

In [ ]:
def trouver_mutations(ref, patient):
    """Compare ref et patient position par position.
    Renvoie la liste des (index, base_ref, base_patient) où elles diffèrent."""
    # 1. créer une liste vide
    # 2. boucle for sur les index : range(len(ref))
    # 3. si ref[i] != patient[i] : ajouter (i, ref[i], patient[i]) à la liste (.append)
    # 4. renvoyer la liste
    pass  # ← remplacez cette ligne par votre code


# 5. appliquer la fonction à adn_wt et adn_patient, ranger le résultat dans `mutations`
# 6. pour chaque différence, afficher l'index Python ET le numéro du nucléotide (index + 1)

In [ ]:
# ✔️ Vérification — exécutez sans modifier
essai = trouver_mutations("ACGT", "ACCT")
if essai == [(2, "G", "C")]:
    print("✅ sur l'exemple ACGT / ACCT :", essai)
else:
    print("❌ sur l'exemple ACGT / ACCT : attendu [(2, 'G', 'C')], obtenu", essai)

if len(mutations) == 1:
    print("✅ une seule différence entre la référence et le patient :", mutations)
else:
    print("❌ attendu 1 différence, trouvé", len(mutations))

## 3. Quel codon, quel acide aminé ?

Le ribosome lit l'ARN trois bases à la fois, à partir du codon start : le nucléotide d'index `i`
appartient au codon d'index `i // 3` (division entière). Les fonctions du notebook 1 donnent les
codons d'une séquence : `codons(transcrire(adn))`.

In [ ]:
# 1. index_codon = index du nucléotide muté (mutations[0][0]) divisé par 3 (division entière //)
# 2. codons_wt = codons(transcrire(adn_wt)), et de même codons_patient
# 3. afficher le numéro du codon muté (index_codon + 1)
# 4. afficher ce codon chez la référence et chez le patient
pass  # ← remplacez cette ligne par votre code

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if index_codon == 6 and codons_wt[6] == "GAG" and codons_patient[6] == "GUG":
    print("✅ codon n°7 (index 6) :", codons_wt[6], "→", codons_patient[6])
else:
    print("❌ la mutation doit tomber dans le codon n°7, c'est-à-dire l'index 6 : GAG → GUG")

Traduisez maintenant les deux séquences en protéines, et comparez-les. Une protéine, comme
l'ADN, est une chaîne de caractères : la fonction `trouver_mutations()` de la section 2 marche
donc aussi sur les protéines.

Le dictionnaire `NOMS` donne, pour chaque lettre, l'abréviation à trois lettres et le nom.

In [ ]:
NOMS = {
    "A": ("Ala", "alanine"),
    "R": ("Arg", "arginine"),
    "N": ("Asn", "asparagine"),
    "D": ("Asp", "aspartate"),
    "C": ("Cys", "cystéine"),
    "Q": ("Gln", "glutamine"),
    "E": ("Glu", "glutamate"),
    "G": ("Gly", "glycine"),
    "H": ("His", "histidine"),
    "I": ("Ile", "isoleucine"),
    "L": ("Leu", "leucine"),
    "K": ("Lys", "lysine"),
    "M": ("Met", "méthionine"),
    "F": ("Phe", "phénylalanine"),
    "P": ("Pro", "proline"),
    "S": ("Ser", "sérine"),
    "T": ("Thr", "thréonine"),
    "W": ("Trp", "tryptophane"),
    "Y": ("Tyr", "tyrosine"),
    "V": ("Val", "valine"),
}

print(NOMS["E"], NOMS["V"])

In [ ]:
# 1. proteine_wt = traduire(transcrire(adn_wt)), et de même proteine_patient
# 2. appliquer trouver_mutations aux deux protéines → `differences`
# 3. pour chaque (index, aa_wt, aa_patient) de differences :
#      afficher « position N : X → Y » avec N = index + 1
#      afficher le nom complet des deux acides aminés avec NOMS[lettre][1]
pass  # ← remplacez cette ligne par votre code

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if len(proteine_wt) == 147 and len(proteine_patient) == 147:
    print("✅ 147 acides aminés de chaque côté")
else:
    print("❌ attendu 147 acides aminés, trouvé", len(proteine_wt), "et", len(proteine_patient))

if differences == [(6, "E", "V")]:
    print("✅ position 7 de la chaîne traduite :", NOMS["E"][1], "→", NOMS["V"][1])
else:
    print("❌ attendu une seule différence, (6, 'E', 'V'), et vous obtenez", differences)

## 4. La mutation change-t-elle beaucoup l'acide aminé ?

Les vingt acides aminés ont le même squelette ; ils diffèrent par leur **chaîne latérale**. On
les range en quatre classes, selon ce que porte cette chaîne latérale :

| Classe | Chaîne latérale | Exemples |
|---|---|---|
| apolaire | uniquement des liaisons C–H | valine, leucine, phénylalanine |
| polaire | un groupe polaire, sans charge (–OH, –NH₂…) | sérine, thréonine, glutamine |
| acide (chargé −) | un groupe –COO⁻ | aspartate, glutamate |
| basique (chargé +) | un groupe chargé + (–NH₃⁺…) | lysine, arginine, histidine |

Un changement **dans la même classe** (glutamate → aspartate, par exemple) modifie peu la
protéine. Un changement **de classe** peut la modifier beaucoup.

Le dictionnaire `CLASSE` donne la classe de chaque acide aminé.

In [ ]:
# classe de la chaîne latérale de chaque acide aminé
CLASSE = {
    "G": "apolaire", "A": "apolaire", "V": "apolaire", "L": "apolaire", "I": "apolaire", "M": "apolaire", "F": "apolaire", "W": "apolaire", "P": "apolaire",
    "S": "polaire", "T": "polaire", "C": "polaire", "Y": "polaire", "N": "polaire", "Q": "polaire",
    "D": "acide (chargé −)", "E": "acide (chargé −)",
    "K": "basique (chargé +)", "R": "basique (chargé +)", "H": "basique (chargé +)",
}

print(CLASSE["E"], "/", CLASSE["V"])

In [ ]:
# 1. index, aa_wt, aa_patient = differences[0] : la différence trouvée à la section 3
# 2. classe_wt = CLASSE[aa_wt], et de même classe_patient ; afficher les deux, avec le nom
#    de l'acide aminé (NOMS[lettre][1])
# 3. meme_classe = (classe_wt == classe_patient) ; l'afficher
pass  # ← remplacez cette ligne par votre code

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if classe_wt == "acide (chargé −)" and classe_patient == "apolaire" and meme_classe == False:
    print("✅ un acide aminé chargé remplacé par un acide aminé apolaire : changement de classe")
else:
    print("❌ attendu : acide (chargé −) puis apolaire, et meme_classe = False ; obtenu",
          classe_wt, "/", classe_patient, "/", meme_classe)

**Question.** Chaîne latérale du glutamate : –CH₂–CH₂–COO⁻. Chaîne latérale de la valine :
–CH(CH₃)₂. Avec la règle « charges dehors, gras dedans » : si cet acide aminé est à la surface
de la protéine, au contact de l'eau, que change la mutation ?

✍️ *Votre réponse (double-cliquez sur cette cellule pour écrire) :*

## 5. Les séquences pour AlphaFold

L'hémoglobine est un **tétramère** : deux chaînes α (gène `HBA1`) et deux chaînes β (gène
`HBB`). Chaque chaîne porte un **hème** : une molécule plane avec, en son centre, un atome de fer
qui fixe une molécule d'O₂.

Dans le globule rouge, la **méthionine initiale** est retirée de chaque chaîne : les chaînes
matures commencent à l'acide aminé suivant. `[1:]` retire le premier caractère d'une chaîne.

In [ ]:
# 1. alpha = la chaîne traduite de REFERENCE["HBA1"], sans son premier acide aminé ([1:])
#    (traduire(transcrire(...)))
# 2. beta_normale : même chose avec REFERENCE["HBB"]
# 3. beta_patient : même chose avec PATIENT["HBB"]
# 4. afficher chaque chaîne et sa longueur
pass  # ← remplacez cette ligne par votre code

In [ ]:
# ✔️ Vérification — exécutez sans modifier
if len(alpha) == 141 and len(beta_normale) == 146 and len(beta_patient) == 146:
    print("✅ chaînes matures : 141 acides aminés pour α, 146 pour β")
else:
    print("❌ attendu 141 (α) et 146 (β) acides aminés, trouvé",
          len(alpha), len(beta_normale), len(beta_patient))

if beta_normale[5] == "E" and beta_patient[5] == "V":
    print("✅ position 6 (index 5) : E chez la normale, V chez le patient — c'est Glu6Val")
else:
    print("❌ attendu E puis V en position 6, trouvé", beta_normale[5], "et", beta_patient[5])

Sans la méthionine initiale, le glutamate muté passe de la position 7 (section 3) à la
**position 6** : c'est la numérotation de la littérature médicale (**Glu6Val**) et des fichiers
de structure.

### Lancer les prédictions sur le serveur AlphaFold

AlphaFold 3 prédit la structure d'un assemblage de molécules — protéines, acides nucléiques,
petites molécules comme l'hème — à partir de la séquence de chaque chaîne. On lui soumet deux
hémoglobines : l'une avec la chaîne β normale, l'autre avec celle du patient.

**Sur [alphafoldserver.com](https://alphafoldserver.com)**, dans un nouvel onglet :

1. connectez-vous avec votre compte Google (à la première connexion, acceptez les conditions
   d'utilisation) ;
2. première entité : *Protein*, collez la séquence `alpha` (affichée plus haut), **2 copies** ;
3. **Add entity** → *Protein*, collez `beta_normale`, **2 copies** ;
4. **Add entity** → *Ligand*, choisissez `HEM` (l'hème), **4 copies** ;
5. **Continue and preview job**, nommez le job `hb_normal`, puis **Confirm and submit job** ;
6. recommencez avec `beta_patient` à la place de `beta_normale`, et nommez le job `hb_patient`.

Pour copier une séquence sans erreur, affichez-la seule : `print(beta_patient)`, puis
sélectionnez-la. Les deux demandes existent aussi toutes faites, en fichiers JSON à importer avec
**Upload JSON**, sur la [page d'accueil des notebooks](https://cedricusureau.github.io/cours-bio-notebooks/).

Chaque prédiction prend quelques minutes.

## 6. Regarder le résultat

Sur le serveur, ouvrez chaque prédiction terminée depuis l'historique des jobs. La structure est
colorée par le **pLDDT**, le score de confiance d'AlphaFold pour chaque résidu, de 0 à 100 :
bleu foncé au-dessus de 90 (très fiable), bleu clair de 70 à 90, jaune de 50 à 70, orange en
dessous de 50.

**Questions.**
1. La prédiction est-elle fiable ?
2. À l'œil, l'hémoglobine du patient a-t-elle une forme différente de l'hémoglobine normale ?

Pour comparer précisément, la cellule suivante superpose les deux prédictions préparées pour le
cours : la normale en gris, celle du patient en bleu, le résidu 6 des deux chaînes β en
bâtonnets (Glu6 en rouge, Val6 en orange), les hèmes en rouge foncé. À droite, zoom sur le
résidu 6 d'une chaîne β. Les vues se manipulent à la souris : glisser pour tourner, molette pour
zoomer.

In [ ]:
#@title ▶️ Exécutez cette cellule pour voir les deux prédictions superposées (ne pas modifier)
# Prédictions AlphaFold 3 préparées pour le cours : le meilleur modèle (model_0) de chaque job.
# Elles sont soumises aux « AlphaFold Server Output Terms of Use » : alphafoldserver.com/output-terms
!pip install -q py3Dmol

import urllib.request

import numpy as np
import py3Dmol

DEPOT = "https://raw.githubusercontent.com/cedricusureau/cours-bio-notebooks/main/data/"


def charger(nom):
    """Le texte du fichier `nom` : copie locale si elle existe, sinon téléchargé depuis le dépôt du cours."""
    try:
        with open(nom) as f:
            return f.read()
    except FileNotFoundError:
        with urllib.request.urlopen(DEPOT + nom, timeout=30) as reponse:
            texte = reponse.read().decode()
        with open(nom, "w") as f:
            f.write(texte)
        return texte


def carbones_alpha(texte):
    """{(chaîne, numéro de résidu): [x, y, z]} pour les carbones α d'un fichier mmCIF."""
    champs, ca = [], {}
    for ligne in texte.splitlines():
        if ligne.startswith("_atom_site."):
            champs.append(ligne.split(".", 1)[1].strip())
        elif ligne.startswith("ATOM") and champs:
            a = dict(zip(champs, ligne.split()))
            if a["label_atom_id"] == "CA":
                ca[(a["auth_asym_id"], a["auth_seq_id"])] = [float(a["Cartn_x"]), float(a["Cartn_y"]), float(a["Cartn_z"])]
    return ca


def deplacer(texte, rotation, centre, cible):
    """Le même fichier, toutes les coordonnées déplacées : (x − centre) · rotation + cible."""
    champs, sortie = [], []
    for ligne in texte.splitlines():
        if ligne.startswith("_atom_site."):
            champs.append(ligne.split(".", 1)[1].strip())
        elif ligne.startswith(("ATOM", "HETATM")) and champs:
            v = ligne.split()
            ix, iy, iz = champs.index("Cartn_x"), champs.index("Cartn_y"), champs.index("Cartn_z")
            p = (np.array([float(v[ix]), float(v[iy]), float(v[iz])]) - centre) @ rotation + cible
            v[ix], v[iy], v[iz] = f"{p[0]:.3f}", f"{p[1]:.3f}", f"{p[2]:.3f}"
            ligne = " ".join(v)
        sortie.append(ligne)
    return "\n".join(sortie)


normale = charger("af3_hb_normal.cif")
patient = charger("af3_hb_patient.cif")

# superposition (algorithme de Kabsch) sur les carbones α des quatre chaînes
ca_n, ca_p = carbones_alpha(normale), carbones_alpha(patient)
communs = sorted(set(ca_n) & set(ca_p))
Q = np.array([ca_n[k] for k in communs])
P = np.array([ca_p[k] for k in communs])
U, _, Vt = np.linalg.svd((P - P.mean(0)).T @ (Q - Q.mean(0)))
signe = np.sign(np.linalg.det(U @ Vt))
rotation = U @ np.diag([1, 1, signe]) @ Vt
patient_superpose = deplacer(patient, rotation, P.mean(0), Q.mean(0))
ecart = np.sqrt((np.linalg.norm((P - P.mean(0)) @ rotation - (Q - Q.mean(0)), axis=1) ** 2).mean())
print("écart moyen entre les deux prédictions, après superposition :", round(ecart, 2), "Å,",
      "sur", len(communs), "carbones α")

# à gauche : tout le tétramère ; à droite : zoom sur le résidu 6 d'une chaîne β (chaîne C)
vue = py3Dmol.view(width=950, height=450, viewergrid=(1, 2), linked=False)
for case in [(0, 0), (0, 1)]:
    zoom = case == (0, 1)
    vue.addModel(normale, "cif", viewer=case)
    vue.addModel(patient_superpose, "cif", viewer=case)
    # dans le zoom, rubans transparents et pas d'hème : le résidu 6 reste visible
    vue.setStyle({"model": 0}, {"cartoon": {"color": "#94a3b8", "opacity": 0.45 if zoom else 1}}, viewer=case)
    vue.setStyle({"model": 1}, {"cartoon": {"color": "#2563eb", "opacity": 0.35 if zoom else 0.6}}, viewer=case)
    if not zoom:
        vue.addStyle({"resn": "HEM"}, {"stick": {"color": "#b91c1c", "radius": 0.2}}, viewer=case)
    for chaine in ["C", "D"]:
        vue.addStyle({"model": 0, "chain": chaine, "resi": 6}, {"stick": {"color": "#f43f5e", "radius": 0.3}}, viewer=case)
        vue.addStyle({"model": 1, "chain": chaine, "resi": 6}, {"stick": {"color": "#d97706", "radius": 0.3}}, viewer=case)
vue.addLabel("Glu6 (normale)", {"fontSize": 12, "backgroundColor": "#f43f5e"},
             {"model": 0, "chain": "C", "resi": 6, "atom": "CD"}, viewer=(0, 1))
vue.addLabel("Val6 (patient)", {"fontSize": 12, "backgroundColor": "#d97706"},
             {"model": 1, "chain": "C", "resi": 6, "atom": "CB"}, viewer=(0, 1))
vue.zoomTo(viewer=(0, 0))
vue.zoomTo({"chain": "C", "resi": list(range(1, 16))}, viewer=(0, 1))
vue.show()

## Bilan

| Question | Réponse |
|---|---|
| Quel gène est muté ? | `HBB`, la bêta-globine |
| Quelle base ? | nucléotide n°20 : `A` → `T` |
| Quel codon ? | codon n°7 : `GAG` → `GUG` |
| Quel acide aminé ? | glutamate → valine, position 6 de la protéine mature (Glu6Val) |
| Est-ce un grand changement ? | oui : un acide aminé chargé remplacé par un acide aminé apolaire |
| La forme change-t-elle ? | non : les deux prédictions se superposent à quelques dixièmes d'ångström près |

**La forme de l'hémoglobine du patient est la même ; ce qui change, c'est un point de sa
surface.** Le résidu 6 est au bord de la chaîne β, au contact de l'eau : la mutation y remplace
une charge par une tache apolaire.

Une seule hémoglobine ne montre pas la maladie : elle vient de ce que les hémoglobines du patient
font **entre elles**. La tache apolaire d'une hémoglobine se colle contre une surface apolaire
d'une hémoglobine voisine, et de proche en proche elles s'assemblent en longues fibres, qui
déforment le globule rouge en faucille.

**Deux limites à connaître.** AlphaFold n'a pas été conçu pour prédire l'effet d'une mutation
ponctuelle : une prédiction identique ne prouverait rien à elle seule. Ici, les structures
mesurées expérimentalement (cristallographie) confirment que la forme ne change pas. Et les
prédictions sont des modèles théoriques, sans usage clinique.